In [2]:
import pandas as pd
import re

# File paths
input_file = "ALL AHN SHIPPED SPECIMENS.xlsx"
output_file = "aliquot_DCC_data_release_v2-0-10_shipped_updated-labs.tsv"

# Read the Excel file
df = pd.read_excel(input_file, dtype=str)

# Remove rows with missing Barcode
df = df[df["Barcode"].notna()]

# Function to generate *follow_ups.submitter_id
def generate_followup_id(row):
    barcode = row["Barcode"]
    study_visit = str(row["Study Visit"]).strip()
    
    if not isinstance(barcode, str) or len(barcode) < 5:
        return ""

    patient_id = barcode[:5]

    if study_visit.startswith("Week"):
        match = re.search(r"Week\s*(\d+)", study_visit)
        if match:
            week_num = int(match.group(1))
            day_equivalent = week_num * 7
            return f"{patient_id}_obs_{day_equivalent}"
    elif study_visit.startswith("Day"):
        match = re.search(r"Day\s*(\d+)", study_visit)
        if match:
            day_num = int(match.group(1))
            return f"{patient_id}_clinical_{day_num}"
    elif study_visit == "Baseline":
        return f"{patient_id}_clinical_0"
    
    return ""

# Apply follow-up ID generation
df["*follow_ups.submitter_id"] = df.apply(generate_followup_id, axis=1)

# Mapping from "Sent to" to labs.submitter_id
sent_to_mapping = {
    "Szabo": "lab_1",
    "Nagy": "lab_2",
    "Liangpunsakul": "lab_3",
    "McClain": "lab_6",
    "Bataller": "lab_7",
    "Mitchell": "lab_8",
    "Malhi": "lab_12",
    "Simonetto": "lab_13",
    "Schnabl": "lab_14",
    "Mehal": "lab_15"
}

# Apply the mapping using "Sent to", default to empty string if not matched
df["labs.submitter_id"] = df["Sent to"].map(sent_to_mapping).fillna("")

# Standardize specimen_type
specimen_mapping = {
    "CPT plasma": "CPT Plasma",
    "HCl sodium citrate plasma": "HCl Sodium Citrate Plasma",
    "NEAT plasma": "NEAT Plasma",
    "Platelet-poor plasma": "Platelet Poor Plasma",
    "Platelet-rich plasma": "Platelet Rich Plasma",
    "Whole blood": "Whole Blood (DNA)"
}
df["specimen_type"] = df["Specimen Type"].replace(specimen_mapping)

# Build the output DataFrame
df_output = pd.DataFrame()
df_output["*type"] = ["aliquot"] * len(df)
df_output["project_id"] = ["ARDaC-AlcHepNet"] * len(df)
df_output["*submitter_id"] = df["Barcode"]
df_output["*follow_ups.submitter_id"] = df["*follow_ups.submitter_id"]
df_output["labs.submitter_id"] = df["labs.submitter_id"]
df_output["aliquot_amount"] = ""
df_output["aliquot_collection_unit"] = ""
df_output["container_type"] = ""
df_output["specimen_type"] = df["specimen_type"]

# Save as TSV
df_output.to_csv(output_file, sep="\t", index=False)